In [8]:


import numpy as np
import xarray as xr
import metpy.calc as mpcalc
import time
import os
import glob
from metpy.units import units
from typing import Tuple, List, Dict
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


EXPERIMENTS = ['CNTL', 'P4K', '4CO2']

In [9]:
# ============ 检查并安装必要的依赖 ============
try:
    import psutil
    print("✅ psutil 已安装")
except ImportError:
    print("⚠️  psutil 未安装，正在安装...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'psutil'])
    import psutil
    print("✅ psutil 安装完成")

print(f"💾 系统总内存: {psutil.virtual_memory().total / 1024 / 1024 / 1024:.2f} GB")
print(f"💾 可用内存: {psutil.virtual_memory().available / 1024 / 1024 / 1024:.2f} GB")

✅ psutil 已安装
💾 系统总内存: 501.87 GB
💾 可用内存: 205.02 GB


In [10]:
# 配置 SLURM 集群以处理大数据
print("="*70)
print("🖥️  Setting up SLURM Cluster for distributed computing")
print("="*70)

USE_SLURM = True  # 设置为 True 以使用 SLURM 集群

if USE_SLURM:
    from dask_jobqueue import SLURMCluster
    from dask.distributed import Client
    
    cluster = SLURMCluster(
        name="divergence_3d",
        queue="shared",
        memory="200GB",  # 每个worker的内存
        cores=256,        # 减少核心数到256（更常见的节点配置）
        processes=2,      # 每个作业1个worker进程
        interface="ib0",
        account="mh1498",  # 您的账户
        walltime="08:00:00",
        scheduler_options={"dashboard_address": ":8788"},
    )
    
    client = Client(cluster)
    cluster.scale(2)  # 启动2个worker，可根据需要调整
    
    print(f"✅ SLURM Cluster initialized")
    print(f"📊 Dashboard: {client.dashboard_link}")
    print(f"🔗 Scheduler: {client.scheduler_info()['address']}")
    print(f"👷 Workers: Scaling to 2")
    print("="*70)
    print("⏳ Waiting for workers to start...")
    # client.wait_for_workers(n_workers=1, timeout=300)  # 等待至少1个worker启动
    print("✅ Workers ready!")
    
else:
    print("⚠️  Running in local mode (no SLURM)")
    client = None

print("="*70)

🖥️  Setting up SLURM Cluster for distributed computing
✅ SLURM Cluster initialized
📊 Dashboard: http://10.128.10.130:39231/status
🔗 Scheduler: tcp://10.128.10.130:43969
👷 Workers: Scaling to 2
⏳ Waiting for workers to start...
✅ Workers ready!


/home/m/m301257/.conda/envs/xianpu/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8788 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39231 instead
  warnings.warn(


In [11]:


# ============ 数据处理函数 ============
def load_data(path: str, var: str, lat_range: tuple = (-15, 15)) -> xr.DataArray:
    """加载并预处理数据"""
    ds = xr.open_dataset(path).sortby('lat').sel(lat=slice(*lat_range))
    return ds[var]

def interpolate_nan(da: xr.DataArray) -> xr.DataArray:
    """填充 NaN 值"""
    return (da.interpolate_na(dim='time', method='nearest')
              .interpolate_na(dim='lat', method='nearest')
              .interpolate_na(dim='lon', method='nearest'))

def compute_grid_spacing(lat: np.ndarray, lon: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """计算网格间距 (m)"""
    R = 6.371e6  # 地球半径
    xlon, ylat = np.meshgrid(lon, lat)
    dlonx = np.gradient(xlon, axis=1)
    dlaty = np.gradient(ylat, axis=0)
    dx = R * np.cos(ylat * np.pi / 180) * dlonx * np.pi / 180
    dy = R * dlaty * np.pi / 180
    return dx, dy

def numpy_divergence(u: xr.DataArray, v: xr.DataArray, dx: np.ndarray, dy: np.ndarray) -> np.ndarray:
    """使用 NumPy 计算散度"""
    du_dx = np.gradient(u, axis=-1) * units('m/s') / dx[np.newaxis, :, :]
    dv_dy = np.gradient(v, axis=-2) * units('m/s') / dy[np.newaxis, :, :]
    return (du_dx + dv_dy).magnitude

def find_matching_pressure_level(available_levels: xr.DataArray, target_level: float) -> float:
    """找到匹配的压力层"""
    matched = [p.item() for p in available_levels if np.isclose(p, target_level, atol=1)]
    if not matched:
        raise ValueError(f"未找到匹配的压力层：{target_level}")
    return matched[0]



def get_file_path(base_dir: str, var: str, model: str) -> str:
    """获取文件路径"""
    pattern = os.path.join(base_dir, var, "sametime", f"{var}_day_{model}_*.nc")
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"未找到文件：{pattern}")
    if len(files) > 1:
        print(f"  ⚠️ 找到多个文件，使用第一个：{files[0]}")
    return files[0]

def save_divergence_data(divergence_results: Dict[str, Dict[str, xr.DataArray]], 
                        output_path: str) -> None:
    """保存散度数据"""
    datasets = {}
    for plev, div_data in divergence_results.items():
        # 为每个变量添加压力层属性
        metpy_da = div_data['metpy'].copy()
        numpy_da = div_data['numpy'].copy()
        
        # 添加压力层作为属性而不是坐标
        metpy_da.attrs['pressure_level'] = f'{plev}Pa'
        numpy_da.attrs['pressure_level'] = f'{plev}Pa'
        
        datasets[f'divergence_metpy_{plev}'] = metpy_da
        datasets[f'divergence_numpy_{plev}'] = numpy_da
    
    ds_out = xr.Dataset(datasets)
    
    # 添加全局属性
    ds_out.attrs['description'] = 'Divergence calculated using MetPy and NumPy methods'
    ds_out.attrs['pressure_levels'] = f'{list(divergence_results.keys())}Pa'
    ds_out.attrs['created_by'] = 'Improved divergence calculation script'
    
    ds_out.to_netcdf(output_path)
    print(f"  ✅ 数据已保存：{output_path}")

def plot_divergence_comparison(nc_path: str, plev: float, save_plot: bool = True) -> None:
    """绘制散度对比图"""
    ds = xr.open_dataset(nc_path)
    model = os.path.basename(nc_path).split("_")[0]
    
    div_metpy = ds[f'divergence_metpy_{plev}'].mean(dim='time')
    div_numpy = ds[f'divergence_numpy_{plev}'].mean(dim='time')
    diff = div_metpy - div_numpy
    
    # 创建图形
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), 
                            subplot_kw={'projection': ccrs.PlateCarree()})
    
    vmax = max(np.abs(div_metpy.max()), np.abs(div_metpy.min()))
    titles = ["MetPy Divergence", "NumPy Divergence", "Difference (MetPy - NumPy)"]
    data_list = [div_metpy, div_numpy, diff]
    vmaxs = [vmax, vmax, vmax/10]
    
    for i, (ax, title, data, vm) in enumerate(zip(axes, titles, data_list, vmaxs)):
        cmap = 'RdBu_r' if i < 2 else 'RdBu'
        im = ax.pcolormesh(data.lon, data.lat, data, cmap=cmap, vmin=-vm, vmax=vm)
        ax.set_title(f"{title} ({model}, {plev/100:.0f}hPa)")
        ax.coastlines()
        ax.set_xticks(np.arange(0, 361, 60), crs=ccrs.PlateCarree())
        ax.set_yticks(np.arange(-15, 16, 5), crs=ccrs.PlateCarree())
        fig.colorbar(im, ax=ax, orientation='horizontal', pad=0.05)
    
    plt.tight_layout()
    
    if save_plot:
        plot_path = nc_path.replace('.nc', f'_{plev//100}hPa_comparison.png')
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        print(f"  📊 图像已保存：{plot_path}")
    
    plt.show()
    
    # 统计分析
    print_statistics(ds[f'divergence_metpy_{plev}'], ds[f'divergence_numpy_{plev}'], plev)

def print_statistics(metpy_data: xr.DataArray, numpy_data: xr.DataArray, plev: float) -> None:
    """打印统计信息"""
    m, n = metpy_data.values.flatten(), numpy_data.values.flatten()
    valid = np.isfinite(m) & np.isfinite(n)
    
    if valid.sum() == 0:
        print(f"  ❌ {plev/100:.0f}hPa: 无有效数据进行比较")
        return
    
    corr, _ = pearsonr(m[valid], n[valid])
    rmse = np.sqrt(mean_squared_error(m[valid], n[valid]))
    bias = np.mean(m[valid] - n[valid])
    
    print(f"  📈 {plev/100:.0f}hPa 统计:")
    print(f"     🔹 相关系数: {corr:.4f}")
    print(f"     🔹 RMSE: {rmse:.4e}")
    print(f"     🔹 Bias: {bias:.4e}")

def extract_model_name(file_path: str) -> str:
    """从文件路径提取模型名称"""
    return os.path.basename(file_path).split("_")[2]

        

In [12]:
# ============ 多层散度计算（优化版：内存管理 + 分批处理） ============
import gc
import psutil

def _ocean(ds):
    fraction = xr.open_dataarray(r'../processed_data/land_mask_2deg.nc')
    return fraction == 0
ocean_mask = _ocean(None)

def get_memory_usage():
    """获取当前内存使用情况"""
    process = psutil.Process()
    mem_info = process.memory_info()
    return mem_info.rss / 1024 / 1024 / 1024  # GB

def calculate_and_save_single_level(data_dir, exp_name, exp_info, target_level, output_dir, ocean_mask=None):
    """
    计算并立即保存单个层次的散度数据（避免内存累积）
    
    Returns:
    --------
    bool : 是否成功处理
    """
    # 构建文件路径
    ua_file = f"{data_dir}{exp_info['ua_dir']}/{exp_name}/ua_lev_{target_level:03d}.nc"
    va_file = f"{data_dir}{exp_info['va_dir']}/{exp_name}/va_lev_{target_level:03d}.nc"
    
    if not os.path.exists(ua_file):
        print(f"❌ 文件不存在，跳过 {exp_name} - Level {target_level}")
        return False
    
    print(f"✅ 开始处理 {exp_name.upper()} - Level {target_level}")
    mem_before = get_memory_usage()
    print(f"   💾 处理前内存: {mem_before:.2f} GB")
    
    try:
        # 1. 使用chunking加载数据（懒加载）
        print(f"   📂 加载风场数据...")
        ua_ds = xr.open_dataset(ua_file, chunks={'time': 100})
        va_ds = xr.open_dataset(va_file, chunks={'time': 100})
        
        ua = ua_ds['ua']
        va = va_ds['va']
        
        # 2. 计算网格间距（只需要一次）
        dx, dy = compute_grid_spacing(ua.lat.values, ua.lon.values)
        
        # 3. 分批计算散度（每次处理100个时间步）
        print(f"   🧮 分批计算散度...")
        time_chunks = range(0, len(ua.time), 100)
        divergence_list = []
        
        for i, start_idx in enumerate(time_chunks):
            end_idx = min(start_idx + 100, len(ua.time))
            print(f"      批次 {i+1}/{len(time_chunks)}: 时间步 {start_idx}-{end_idx}")
            
            # 加载这批数据到内存
            ua_chunk = ua.isel(time=slice(start_idx, end_idx)).load()
            va_chunk = va.isel(time=slice(start_idx, end_idx)).load()
            
            # 计算散度
            div_chunk = numpy_divergence(ua_chunk, va_chunk, dx, dy)
            divergence_list.append(div_chunk)
            
            # 及时清理
            del ua_chunk, va_chunk, div_chunk
            gc.collect()
        
        # 4. 合并结果
        print(f"   🔗 合并散度结果...")
        divergence = np.concatenate(divergence_list, axis=0)
        del divergence_list
        gc.collect()
        
        print(f"      散度 shape: {divergence.shape}")
        print(f"      散度范围: {np.nanmin(divergence):.2e} - {np.nanmax(divergence):.2e} s^-1")
        
        # 5. 创建DataArray
        div_da = xr.DataArray(
            divergence,
            dims=('time', 'lat', 'lon'),
            coords={
                'time': ua.time,
                'lat': ua.lat,
                'lon': ua.lon
            },
            attrs={
                'long_name': f'Horizontal divergence at Level {target_level} - Ocean only',
                'units': 's^-1',
                'level': target_level,
                'method': 'NumPy gradient method (vectorized)',
                'domain': 'Ocean only' if ocean_mask is not None else 'Global',
                'experiment': exp_name.upper()
            }
        )
        
        # 应用ocean mask
        if ocean_mask is not None:
            div_da = div_da.where(ocean_mask)
        
        # 6. 立即保存到文件（避免内存累积）
        os.makedirs(output_dir, exist_ok=True)
        output_file = os.path.join(output_dir, f'divergence_lev{target_level}_{exp_name}.nc')
        
        print(f"   💾 保存到文件: {output_file}")
        ds_out = xr.Dataset({'divergence': div_da})
        ds_out.attrs.update({
            'title': f'Divergence at Level {target_level} for {exp_name.upper()} experiment',
            'level': target_level,
            'experiment': exp_name.upper(),
            'method': 'NumPy gradient method',
            'created_date': time.strftime('%Y-%m-%d %H:%M:%S'),
        })
        if not os.path.exists(output_file):
            ds_out.to_netcdf(output_file)
            ds_out.close()
        else:
            print(f"   ⚠️ 文件已存在，跳过保存: {output_file}")
        # 7. 清理内存
        ua_ds.close()
        va_ds.close()
        del ua, va, ua_ds, va_ds, divergence, div_da, ds_out
        gc.collect()
        
        mem_after = get_memory_usage()
        print(f"   ✅ 处理完成！内存释放: {mem_before:.2f} GB -> {mem_after:.2f} GB")
        
        return True
        
    except Exception as e:
        print(f"   ❌ 处理失败: {str(e)}")
        # 确保清理内存
        try:
            ua_ds.close()
            va_ds.close()
        except:
            pass
        gc.collect()
        return False


def calculate_low_level_divergence_multi_levels_optimized(
    data_dir="/work/mh1498/m301257/converted_data/",
    target_levels=[80, 81, 55, 50],
    output_dir='/work/mh1498/m301257/3D_data/divergence/'
):
    """
    优化版多层散度计算 - 逐个处理并保存，避免内存溢出
    
    优化策略：
    1. 逐层逐实验处理（不同时加载多个数据集）
    2. 使用chunking进行懒加载
    3. 分批计算散度
    4. 计算完立即保存，不在内存中累积
    5. 及时释放内存并强制垃圾回收
    """
    print("\n" + "="*70)
    print("🚀 开始优化版多层散度计算")
    print(f"📊 总共需要处理: {len(target_levels)} 个层次 × 3 个实验 = {len(target_levels)*3} 个数据集")
    print(f"💾 初始内存使用: {get_memory_usage():.2f} GB")
    print("="*70)
    
    # 实验配置
    experiments = {
        'cntl': {'ua_dir': 'ua_3d', 'va_dir': 'va_3d'},
        'p4k': {'ua_dir': 'ua_3d', 'va_dir': 'va_3d'},
        '4co2': {'ua_dir': 'ua_3d', 'va_dir': 'va_3d'}
    }
    
    # 统计成功/失败
    success_count = 0
    failed_list = []
    
    # 逐层逐实验处理
    for idx, target_level in enumerate(target_levels):
        print(f"\n{'='*70}")
        print(f"📍 [{idx+1}/{len(target_levels)}] 处理层次: Level {target_level}")
        print(f"{'='*70}")
        
        for exp_name, exp_info in experiments.items():
            success = calculate_and_save_single_level(
                data_dir=data_dir,
                exp_name=exp_name,
                exp_info=exp_info,
                target_level=target_level,
                output_dir=output_dir,
                ocean_mask=ocean_mask
            )
            
            if success:
                success_count += 1
            else:
                failed_list.append((exp_name, target_level))
            
            # 每处理完一个，强制垃圾回收
            gc.collect()
            print(f"   💾 当前内存使用: {get_memory_usage():.2f} GB\n")
    
    # 最终统计
    print("\n" + "="*70)
    print("📊 处理完成统计")
    print(f"✅ 成功: {success_count} 个")
    print(f"❌ 失败: {len(failed_list)} 个")
    if failed_list:
        print("失败列表:")
        for exp, lev in failed_list:
            print(f"   - {exp.upper()} Level {lev}")
    print(f"💾 最终内存使用: {get_memory_usage():.2f} GB")
    print("="*70)
    
    return success_count, failed_list


# 使用优化版本
TARGET_LEVELS = [41,46,51,55,58,63,67,71,74,76,78,80,81,83,84,85,87]


success_count, failed_list = calculate_low_level_divergence_multi_levels_optimized(
    data_dir="/work/mh1498/m301257/converted_data/",
    target_levels=TARGET_LEVELS,
    output_dir='/work/mh1498/m301257/3D_data/divergence/'
)




🚀 开始优化版多层散度计算
📊 总共需要处理: 17 个层次 × 3 个实验 = 51 个数据集
💾 初始内存使用: 0.55 GB

📍 [1/17] 处理层次: Level 41
✅ 开始处理 CNTL - Level 41
   💾 处理前内存: 0.55 GB
   📂 加载风场数据...
   🧮 分批计算散度...
      批次 1/52: 时间步 0-100
      批次 2/52: 时间步 100-200
      批次 3/52: 时间步 200-300
      批次 4/52: 时间步 300-400
      批次 5/52: 时间步 400-500
      批次 6/52: 时间步 500-600
      批次 7/52: 时间步 600-700
      批次 8/52: 时间步 700-800
      批次 9/52: 时间步 800-900
      批次 10/52: 时间步 900-1000
      批次 11/52: 时间步 1000-1100
      批次 12/52: 时间步 1100-1200
      批次 13/52: 时间步 1200-1300
      批次 14/52: 时间步 1300-1400
      批次 15/52: 时间步 1400-1500
      批次 16/52: 时间步 1500-1600
      批次 17/52: 时间步 1600-1700
      批次 18/52: 时间步 1700-1800
      批次 19/52: 时间步 1800-1900
      批次 20/52: 时间步 1900-2000
      批次 21/52: 时间步 2000-2100
      批次 22/52: 时间步 2100-2200
      批次 23/52: 时间步 2200-2300
      批次 24/52: 时间步 2300-2400
      批次 25/52: 时间步 2400-2500
      批次 26/52: 时间步 2500-2600
      批次 27/52: 时间步 2600-2700
      批次 28/52: 时间步 2700-2800
      批次 29/52: 时间步 2800

In [13]:
# ============ 验证保存的文件 ============

def verify_saved_files(output_dir='/work/mh1498/m301257/3D_data/divergence/', target_levels=None):
    """
    验证已保存的散度文件
    """
    print("\n" + "="*70)
    print("📋 验证已保存的散度文件")
    print("="*70)
    
    if not os.path.exists(output_dir):
        print(f"❌ 输出目录不存在: {output_dir}")
        return
    
    saved_files = sorted(glob.glob(os.path.join(output_dir, 'divergence_lev*.nc')))
    
    if not saved_files:
        print(f"⚠️  未找到任何散度文件在: {output_dir}")
        return
    
    print(f"✅ 找到 {len(saved_files)} 个文件\n")
    
    # 按实验和层次统计
    experiments = {'cntl': [], 'p4k': [], '4co2': []}
    
    for f in saved_files:
        filename = os.path.basename(f)
        # 提取层次和实验名
        # 格式: divergence_lev80_cntl.nc
        parts = filename.replace('.nc', '').split('_')
        level = int(parts[1].replace('lev', ''))
        exp = parts[2]
        
        if exp in experiments:
            experiments[exp].append(level)
        
        # 快速检查文件
        try:
            ds = xr.open_dataset(f)
            div_shape = ds['divergence'].shape
            ds.close()
            print(f"   ✓ {filename:40s} shape: {div_shape}")
        except Exception as e:
            print(f"   ✗ {filename:40s} 错误: {str(e)}")
    
    # 统计各实验的层次
    print("\n" + "-"*70)
    print("📊 各实验已保存的层次统计:")
    for exp, levels in experiments.items():
        if levels:
            levels_sorted = sorted(levels)
            print(f"   {exp.upper():5s}: {len(levels)} 层 -> {levels_sorted}")
        else:
            print(f"   {exp.upper():5s}: 无数据")
    
    # 检查缺失的层次
    if target_levels:
        print("\n" + "-"*70)
        print("🔍 检查缺失的层次:")
        for exp, levels in experiments.items():
            missing = set(target_levels) - set(levels)
            if missing:
                print(f"   {exp.upper()}: 缺失 {sorted(missing)}")
            else:
                print(f"   {exp.upper()}: ✓ 完整")
    
    print("="*70)

# 验证文件
verify_saved_files(
    output_dir='/work/mh1498/m301257/3D_data/divergence/',
    target_levels=TARGET_LEVELS
)


📋 验证已保存的散度文件
✅ 找到 51 个文件

   ✓ divergence_lev41_4co2.nc                 shape: (5114, 15, 180)
   ✓ divergence_lev41_cntl.nc                 shape: (5114, 15, 180)
   ✓ divergence_lev41_p4k.nc                  shape: (5114, 15, 180)
   ✓ divergence_lev46_4co2.nc                 shape: (5114, 15, 180)
   ✓ divergence_lev46_cntl.nc                 shape: (5114, 15, 180)
   ✓ divergence_lev46_p4k.nc                  shape: (5114, 15, 180)
   ✓ divergence_lev51_4co2.nc                 shape: (5114, 15, 180)
   ✓ divergence_lev51_cntl.nc                 shape: (5114, 15, 180)
   ✓ divergence_lev51_p4k.nc                  shape: (5114, 15, 180)
   ✓ divergence_lev55_4co2.nc                 shape: (5114, 15, 180)
   ✓ divergence_lev55_cntl.nc                 shape: (5114, 15, 180)
   ✓ divergence_lev55_p4k.nc                  shape: (5114, 15, 180)
   ✓ divergence_lev58_4co2.nc                 shape: (5114, 15, 180)
   ✓ divergence_lev58_cntl.nc                 shape: (5114, 15, 180)
   ✓ di